# 探索：在不熟悉的代码库中立足事实

本 Notebook 把智能体放进一个它从未见过的仓库，要求它弄清真实的架构。文件系统是智能体唯一的工作区，只有它主动读取的文件才会进入上下文窗口，因此它靠 `ls`、`grep` 和 `read` 的探索来建立起心智模型。

有意思的地方在于我们在测试夹具中埋下的一个陷阱。`ARCHITECTURE.md` 描述的布局代码早已不再遵循，因此只信文档而不核对代码的智能体会自信地给出错误答案。在这个语境下，立足事实意味着把读到的内容与仓库中实际存在的东西相互验证，而不是把文档当作权威。

相比 iterate 那份 Notebook，本例还能教会：

- **先探索，再行动。** 好的智能体会先读足够多的目录树以理解它，然后再作答，而不是反过来。
- **在会话进行中追加资源。** 结尾的附录演示如何通过 `sessions.resources.add` 把更多文件推入正在运行的会话，而不是重建会话。当探索发现智能体接下来应该查看的内容时，这一点很有用。

In [ ]:
import io
import os

from anthropic import Anthropic
from utilities import (
    make_unfamiliar_repo_zip,
    stream_until_end_turn,
    wait_for_idle_status,
)

MODEL = os.environ.get("COOKBOOK_MODEL", "claude-sonnet-4-6")

client = Anthropic()

## 1. 生成仓库夹具

仓库足够小，我们用辅助函数在内存中构建它，而不必在 Notebook 旁保留磁盘夹具。该辅助函数会植入一个 `services/` 微服务布局，以及一份仍在描述旧单体布局的过期 `ARCHITECTURE.md`。

In [ ]:
buf = make_unfamiliar_repo_zip()
fixture_zip = client.beta.files.upload(file=("repo.zip", buf, "application/zip"))
print(f"fixture: {fixture_zip.id}")

## 2. 智能体 + 环境 + 会话

In [ ]:
agent = client.beta.agents.create(
    name="cookbook-explore",
    # Tells Anthropic which cookbook this agent came from. Safe to remove.
    metadata={"anthropic_cookbook": "claude-cookbooks/cma-explore-unfamiliar-codebase"},
    model=MODEL,
    system=(
        "You are onboarding to an unfamiliar codebase. Explore before "
        "answering, docs can be stale. Verify what you read against "
        "actual code structure. Write notes to /tmp/NOTES.md as you go."
    ),
    tools=[
        {
            "type": "agent_toolset_20260401",
            "default_config": {
                "enabled": True,
                "permission_policy": {"type": "always_allow"},
            },
        }
    ],
)

env = client.beta.environments.create(
    name="cookbook-explore-env",
    config={"type": "cloud", "networking": {"type": "limited"}},
)

session = client.beta.sessions.create(
    environment_id=env.id,
    agent={"type": "agent", "id": agent.id, "version": agent.version},
    resources=[{"type": "file", "file_id": fixture_zip.id, "mount_path": "repo.zip"}],
    title="Onboard to repo",
)
print(f"session: {session.id}")

## 3. 探索，并留意过期文档陷阱

立足事实的回答会提到真实的 `services/` 布局，并指出 `ARCHITECTURE.md` 已过时。未立足事实的回答则会照搬过期文档所描述的单体布局。

In [ ]:
client.beta.sessions.events.send(
    session_id=session.id,
    events=[
        {
            "type": "user.message",
            "content": [
                {
                    "type": "text",
                    "text": (
                        "Unzip /mnt/session/uploads/repo.zip to /tmp/repo/. "
                        "Then: what is the actual architecture of this "
                        "codebase? Be specific about directory structure. "
                        "Check if the docs are accurate."
                    ),
                }
            ],
        }
    ],
)

print("=== exploration ===")
stream_until_end_turn(client, session.id)

## 4. 回读智能体的笔记

我们事先告诉智能体在工作过程中把笔记写在 `/tmp/NOTES.md`。打印该文件是观察它在探索过程中如何逐步理解代码库的有效方式。

In [ ]:
client.beta.sessions.events.send(
    session_id=session.id,
    events=[
        {
            "type": "user.message",
            "content": [{"type": "text", "text": "cat /tmp/NOTES.md"}],
        }
    ],
)
stream_until_end_turn(client, session.id)

## 附录：为运行中的会话追加上下文

`sessions.create` 上的 `resources=` 参数是最常见的挂载文件方式，但 API 还暴露了 `/v1/sessions/<id>/resources` 子资源，用于管理已有会话上的挂载。这在本例中很有用：如果探索发现某个问题需要额外上下文（一个配置文件、一份变更日志、一个外部模式），可以直接补进去，而不必拆掉会话重来。

套路仍是大家熟悉的"先上传、再挂载"循环，只是拆成两次调用而非一次：

In [ ]:
hints = b"# DEPLOY HISTORY\n2026-03-01: monolith -> microservices migration complete\n"
hints_file = client.beta.files.upload(
    file=("DEPLOY_HISTORY.md", io.BytesIO(hints), "text/markdown")
)

added = client.beta.sessions.resources.add(
    session_id=session.id,
    type="file",
    file_id=hints_file.id,
    mount_path="DEPLOY_HISTORY.md",
)
print(f"added resource {added.id} to session {session.id}")

attached = client.beta.sessions.resources.list(session_id=session.id)
print(f"{len(attached.data)} resources attached now")

client.beta.sessions.events.send(
    session_id=session.id,
    events=[
        {
            "type": "user.message",
            "content": [
                {
                    "type": "text",
                    "text": (
                        "There's a DEPLOY_HISTORY.md in your workspace now. "
                        "Read it and tell me whether it changes anything in "
                        "your earlier answer."
                    ),
                }
            ],
        }
    ],
)
print("\n--- follow-up with deploy history ---")
stream_until_end_turn(client, session.id)

# Detach the file now that the agent is done with it. `delete` here
# is the resource-detach verb, not the cookbook-wide archive.
client.beta.sessions.resources.delete(session_id=session.id, resource_id=added.id)
print("detached follow-up resource")

## 清理

In [ ]:
wait_for_idle_status(client, session.id)
client.beta.sessions.archive(session.id)
client.beta.environments.archive(env.id)
client.beta.agents.archive(agent.id)
print("archived")